## Making initial workflows 

`trovolut` works by creating initial workflows which then can be optimized using `openevolve`. The `filemaker` module contains tools to create these initial workflows. For example, we can define code to run Trotter dynamics with a predefined product formula function in the `dynamics` module for a given backend.

In [3]:
import trovolut as trv
import warnings
warnings.filterwarnings("ignore")

In [4]:
def run_dynamics(Hs, t, psi0):
    """
    Args:
        Hs: list of Hamiltonian terms (numpy arrays)
        t: total time
        psi0: initial state (numpy array)
    """
    r = 10
    U_trotter = trv.backends.cpu.dynamics.product_formula(Hs, t, r)
    psi_final = U_trotter @ psi0
    return psi_final

We can then use the `trv.filemaker.Workflow` class to create a file with these functions. We use the `trv.filemaker.EVO_START` and `trv.filemaker.EVO_END` markers to indicate where `openevolve` should optimize the code.

In [5]:
init_program = trv.filemaker.Workflow(
    [
        trv.filemaker.EVO_START,
        run_dynamics,
        trv.filemaker.EVO_END,
    ],
    out_path="opt/init_program.py",
).build()

print("Wrote", init_program)

Wrote opt/init_program.py


In [6]:
from IPython.display import display, Markdown
import re

# Preview the generated code
display(Markdown(f"### Preview of `{init_program}`:"))
lines = re.findall(r'^(?:#.*|import .+|from .+|def .+:)', open(init_program).read(), re.M)
preview = sum(([line, "    ..."] if line.lstrip().startswith("def ") else [line] for line in lines), [])
display(Markdown("```python\n" + "\n".join(preview) + "\n```"))

### Preview of `opt/init_program.py`:

```python
import numpy as np
import scipy
import trovolut as trv
# EVOLVE-BLOCK-START
def run_dynamics(Hs, t, psi0):
    ...
# EVOLVE-BLOCK-END
```

We define an evaluation function that compares the time evolution to the exact solution (and penalize if the code takes longer than one minute to run).

In [7]:
def get_hamiltonian():
    from rdkit import Chem
    from rdkit.Chem import AllChem

    mol = Chem.AddHs(Chem.MolFromSmiles("CSSC"))
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.UFFOptimizeMolecule(mol)

    conf = mol.GetConformer()
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    xyz = (
        f"{len(atoms)}\ndmds generated by RDKit\n"
        + "\n".join(
            f"{s} {conf.GetAtomPosition(i).x:.8f} "
            f"{conf.GetAtomPosition(i).y:.8f} "
            f"{conf.GetAtomPosition(i).z:.8f}"
            for i, s in enumerate(atoms)
        )
    )

    H = trv.qm.ElectronicHamiltonian.from_xyz(
        xyz_source=xyz,
        basis="sto-3g",
        charge=0,
        spin_mult=1,
        use_gpu4pyscf=False,
        freeze_core=True,
        n_core_keep=2,
        n_virt_keep=1,
    ).to_qubits()

    return [t["coef"] * t["matrix"] for t in H.terms()]

hams = get_hamiltonian()[1:]  # remove identity term
print("Hamiltonian terms:", len(hams))
print("Size of hamiltonian matrices:", hams[0].shape)

converged SCF energy = -864.654938672562
Hamiltonian terms: 81
Size of hamiltonian matrices: (4096, 4096)


In [8]:
import numpy as np
import scipy
import concurrent.futures

def trotter_acc(path):
    
    # compare run_dynamics vs exact solution
    import importlib.util
    spec = importlib.util.spec_from_file_location("init_program", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    
    # parameters
    t, trials = 100, 10
    hams = get_hamiltonian()[1:]  # remove identity term

    # dynamics
    U_exact = scipy.linalg.expm(-1j * sum(hams) * t)

    dim = U_exact.shape[0]
    psi0_batch = np.random.rand(trials, dim) + 1j * np.random.rand(trials, dim)
    psi0_batch = psi0_batch / np.linalg.norm(psi0_batch, axis=1)[:, np.newaxis]

    psi_exact_batch = psi0_batch @ U_exact.T

    psi_trotter_batch = np.array([
        module.run_dynamics(hams, t, psi0_batch[i])
        for i in range(trials)
    ])

    # compute errors
    err = np.linalg.norm(psi_trotter_batch - psi_exact_batch, axis=1) 
    errs = err.tolist()

    # compute accuracy and its std
    mean_err = np.mean(errs)
    acc = 1.0 / (1.0 + mean_err)

    var_err = np.var(errs) / trials
    std_acc = np.sqrt(var_err) / (1.0 + mean_err)**2

    return {"combined_score": acc, "std_score": std_acc}



def run_with_timeout(func, args=(), kwargs=None, timeout_seconds=30):
    if kwargs is None:
        kwargs = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(func, *args, **kwargs)
        try:
            return future.result(timeout=timeout_seconds)
        except concurrent.futures.TimeoutError:
            raise TimeoutError(f"Function timed out after {timeout_seconds} seconds")


def evaluate(program_path):
    """
    OpenEvolve evaluator for Trotter accuracy with a hard timeout.
    If evaluation takes longer than TIMEOUT_SECONDS, return a big penalty score.
    """
    try:
        result = run_with_timeout(
            trotter_acc,  
            args=(program_path,),
            timeout_seconds=30,
        )

        combined = float(result.get("combined_score", -100.0))
        std = float(result.get("std_score", 0.0))

        return {
            "combined_score": combined,
            "std_score": std,
        }

    except Exception as e:
        print(f"Evaluation crashed: {e}")
        return {
            "combined_score": -100.0,
            "std_score": 0.0,
        }

acc = evaluate(init_program)
print(f"Trotter accuracy: {acc['combined_score']} ± {acc['std_score']}")

converged SCF energy = -864.654938666712
Evaluation crashed: Function timed out after 30 seconds
Trotter accuracy: -100.0 ± 0.0


We create another file with such an evaluation function.

In [9]:
def main_eval():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--program_path", type=str, required=True)
    args = parser.parse_args()
    result = evaluate(args.program_path)
    print(f"combined_score: {result['combined_score']}")
    print(f"std_score: {result['std_score']}")


evaluator_path = trv.filemaker.Workflow(
    [
        get_hamiltonian,
        trotter_acc,
        run_with_timeout,
        evaluate
    ],
    out_path="opt/evaluator.py",
    imports=[
        "import concurrent.futures",
        "import numpy as np",
        "import scipy",
        "import trovolut as trv"
    ],
    main_func=main_eval,
).build()

In [10]:
import subprocess

result = subprocess.run(
    ["uv", "run", "python", str(evaluator_path), "--program_path", str(init_program)],
    capture_output=True,
    text=True
)

print(result.stdout)

converged SCF energy = -864.654938672562
Evaluation crashed: Function timed out after 30 seconds
combined_score: -100.0
std_score: 0.0



Now, feeding it into `openevolve`, we can optimize the workflow.

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()

from openevolve import run_evolution
from openevolve.config import Config, LLMModelConfig, PromptConfig

config = Config()
prompt = PromptConfig(system_message="You are an expert programmer specializing in trotterization for quantum evolution. Your task is to maximize the Trotter simulation accuracy of the provided code, but the implementation must run in less than one minute.")

config.prompt = prompt

# If you use OpenAI's API, you can use the following line:
# config.llm.models = [LLMModelConfig(name="gpt-4o")]

# If you use NVIDIA's API, you can use the following line:
config.llm.models = [
    LLMModelConfig(
        name="openai/openai/gpt-5.2",
        api_base="https://inference-api.nvidia.com/v1",
        api_key=os.getenv("NVIDIA_API_KEY"), 
        temperature=0.7,
        max_tokens=1024
    )
]

config.evaluator.cascade_evaluation = False

# Optimize Trotter circuit using OpenEvolve
result = run_evolution(
    initial_program=init_program,
    evaluator=evaluator_path,
    iterations=10,
    config=config
)


2026-02-02 13:40:32,512 - INFO - Logging to /var/folders/q9/p9vhf7d15g9gystjrv_j1pl80000gn/T/openevolve_ck95op6o/logs/openevolve_20260202_134032.log
2026-02-02 13:40:32,512 - INFO - Logging to /var/folders/q9/p9vhf7d15g9gystjrv_j1pl80000gn/T/openevolve_ck95op6o/logs/openevolve_20260202_134032.log
2026-02-02 13:40:32,512 - INFO - Set random seed to 42 for reproducibility
2026-02-02 13:40:32,512 - INFO - Set random seed to 42 for reproducibility
2026-02-02 13:40:32,802 - INFO - Initialized OpenAI LLM with model: openai/openai/gpt-5.2
2026-02-02 13:40:32,802 - INFO - Initialized OpenAI LLM with model: openai/openai/gpt-5.2
2026-02-02 13:40:32,803 - INFO - Initialized LLM ensemble with models: openai/openai/gpt-5.2 (weight: 1.00)
2026-02-02 13:40:32,803 - INFO - Initialized LLM ensemble with models: openai/openai/gpt-5.2 (weight: 1.00)
2026-02-02 13:40:32,842 - INFO - Initialized prompt sampler
2026-02-02 13:40:32,842 - INFO - Initialized prompt sampler
2026-02-02 13:40:32,844 - INFO - Set

converged SCF energy = -864.654938704692
Evaluation crashed: Function timed out after 30 seconds


2026-02-02 13:41:06,650 - INFO - Evaluated program 69afc2b4-d61c-4167-8ba5-b6e6740d0d5f in 33.79s: combined_score=-100.0000, std_score=0.0000
2026-02-02 13:41:06,650 - INFO - Evaluated program 69afc2b4-d61c-4167-8ba5-b6e6740d0d5f in 33.79s: combined_score=-100.0000, std_score=0.0000
2026-02-02 13:41:06,655 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 5, 'diversity': 0}
2026-02-02 13:41:06,655 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 5, 'diversity': 0}
2026-02-02 13:41:06,689 - INFO - Initialized process parallel controller with 1 workers
2026-02-02 13:41:06,689 - INFO - Initialized process parallel controller with 1 workers
2026-02-02 13:41:06,690 - INFO - Set max None tasks per child
2026-02-02 13:41:06,690 - INFO - Set max None tasks per child
2026-02-02 13:41:06,691 - INFO - Started process pool with 1 processes
2026-02-02 13:41:06,691 - INFO - Started process pool with 1 processes
2026-02-02 13:41:06,692 - INFO - Using island-based evol

converged SCF energy = -864.654938672562
Evaluation crashed: Function timed out after 30 seconds


2026-02-02 13:41:50,755 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 9, 'diversity': 5}
2026-02-02 13:41:50,755 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 9, 'diversity': 5}
2026-02-02 13:41:50,757 - INFO - Iteration 1: Program 1345df3f-305d-4de4-9975-d4840aea6c86 (parent: 69afc2b4-d61c-4167-8ba5-b6e6740d0d5f) completed in 40.08s
2026-02-02 13:41:50,757 - INFO - Iteration 1: Program 1345df3f-305d-4de4-9975-d4840aea6c86 (parent: 69afc2b4-d61c-4167-8ba5-b6e6740d0d5f) completed in 40.08s
2026-02-02 13:41:50,758 - INFO - Metrics: combined_score=-100.0000, std_score=0.0000
2026-02-02 13:41:50,758 - INFO - Metrics: combined_score=-100.0000, std_score=0.0000


converged SCF energy = -864.654938666712


2026-02-02 13:42:33,269 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 9, 'diversity': 9}
2026-02-02 13:42:33,269 - INFO - New MAP-Elites cell occupied in island 0: {'complexity': 9, 'diversity': 9}
2026-02-02 13:42:33,271 - INFO - Iteration 2: Program 2d7efedf-8519-4956-a827-656f1ddc658d (parent: 69afc2b4-d61c-4167-8ba5-b6e6740d0d5f) completed in 42.51s
2026-02-02 13:42:33,271 - INFO - Iteration 2: Program 2d7efedf-8519-4956-a827-656f1ddc658d (parent: 69afc2b4-d61c-4167-8ba5-b6e6740d0d5f) completed in 42.51s
2026-02-02 13:42:33,271 - INFO - Metrics: combined_score=-100.0000, std_score=0.0000
2026-02-02 13:42:33,271 - INFO - Metrics: combined_score=-100.0000, std_score=0.0000


Evaluation crashed: Function timed out after 30 seconds
converged SCF energy = -864.654938704692


2026-02-02 13:45:01,349 - INFO - Received signal 2, initiating graceful shutdown...
2026-02-02 13:45:01,349 - INFO - Received signal 2, initiating graceful shutdown...
2026-02-02 13:45:01,352 - INFO - Graceful shutdown requested...
2026-02-02 13:45:01,352 - INFO - Graceful shutdown requested...
2026-02-02 13:45:01,353 - INFO - Shutdown requested, canceling remaining evaluations...
2026-02-02 13:45:01,353 - INFO - Shutdown requested, canceling remaining evaluations...
2026-02-02 13:45:01,354 - INFO - ✅ Evolution completed - Shutdown requested
2026-02-02 13:45:01,354 - INFO - ✅ Evolution completed - Shutdown requested
2026-02-02 13:45:01,355 - INFO - Evolution stopped due to shutdown request
2026-02-02 13:45:01,355 - INFO - Evolution stopped due to shutdown request


After mutating the code for 10 cycles, we get the following optimized trotterization code:

In [ ]:
display(Markdown(f"```python\n{result.best_code}\n```"))

```python
import numpy as np
import scipy
import trovolut as trv

# EVOLVE-BLOCK-START

def product_formula(H, t, r):
    """Second-order Trotter-Suzuki decomposition."""
    dt = t / r
    U_trotter = np.eye(H[0].shape[0], dtype=complex)

    for _ in range(r):
        for H_j in H:
            U_j_half = scipy.linalg.expm(-0.5j * H_j * dt)
            U_trotter = U_j_half @ U_trotter @ U_j_half

    return U_trotter

def run_dynamics(mats, t, psi0):
    """
    Args:
        mats: list of Hamiltonian terms (numpy arrays)
        t: total time
        psi0: initial state (numpy array)
    """
    r = 20  # Increase r for better accuracy
    U_trotter = product_formula(mats, t, r)
    psi_final = U_trotter @ psi0
    return psi_final

# EVOLVE-BLOCK-END


```

In [ ]:
# save best program
with open("opt/best_trotter_program.py", "w") as f:
    f.write(result.best_code)